In [1]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv, find_dotenv
from typing import TypedDict
from langchain_openai import ChatOpenAI
import os

In [2]:
load_dotenv(find_dotenv())

True

In [3]:
model = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://models.inference.ai.azure.com",
    api_key=os.getenv("GITHUB_TOKEN")
)

In [4]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [6]:
def create_outline(state: BlogState) -> BlogState:
    topic = state['title']
    prompt = f'Generate a detailed outline for a blog, on the topic {topic}'
    outline = model.invoke(prompt).content
    state['outline'] = outline
    return state

def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']
    prompt = f'Write a detailed blog on the title {title}, using the outline {outline}'
    content = model.invoke(prompt).content
    state['content'] = content
    return state

In [8]:
graph = StateGraph(BlogState)

graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)

graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", END)

workflow = graph.compile()

In [9]:
initial_state = {'title': "Rise of AI in India"}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': "# Blog Outline: The Rise of AI in India\n\n## Introduction\n- Brief overview of AI and its significance\n- Contextual background on India's technological landscape\n- Purpose of the blog: to explore the rise of AI in India, its impact on various sectors, benefits, challenges, and future prospects\n\n## Section 1: Understanding AI\n- Definition of Artificial Intelligence\n- Key components of AI (Machine Learning, Natural Language Processing, Robotics, etc.)\n- Brief history of AI development globally\n\n## Section 2: Evolution of AI in India\n- Early adaptations of AI technologies in India\n- Government initiatives and policies promoting AI (e.g., NITI Aayog's National Strategy on AI)\n- Growth of AI startups and innovation hubs\n\n## Section 3: Current AI Landscape in India\n- Prominent AI sectors in India\n  - Healthcare: Applications of AI in diagnostics and patient care\n  - Agriculture: Use of AI in precision farming and crop management\

In [10]:
print(final_state['content'])

# The Rise of AI in India

## Introduction

Artificial Intelligence (AI) has burgeoned into a transformative force within the realm of technology, profoundly influencing diverse industries and shaping the way we live, work, and interact. As one of the world's fastest-growing economies, India's technological landscape is a unique tapestry woven with innovation, ambition, and strategic governmental initiatives. The purpose of this blog is to delve into the remarkable rise of AI in India, examining its profound impact on various sectors, the benefits it offers, the challenges that accompany its growth, and its future prospects.

## Section 1: Understanding AI

Artificial Intelligence can be defined as the simulation of human intelligence processes by machines, particularly computer systems. Key components of AI include:

- **Machine Learning (ML)**: A subset of AI that allows systems to learn and improve from experience without explicit programming.
- **Natural Language Processing (NLP)**